# Verifying Quicopt's Gset Max-Cut solutions

This notebook independently checks every published solution **certificate** in
[`../solutions/gset.json`](../solutions/gset.json). For each Gset instance it recomputes the
cut directly from the graph and confirms it equals both the certificate's stated value and the
published leaderboard ([`../../data/gset.csv`](../../data/gset.csv)).

**What a certificate is.** One bitstring per instance — the single best cut Quicopt found.
`bits[i]='1'` means spin $x_i=+1$, `'0'` means $x_i=-1$; the two values label the two sides of
the partition. The cut is

$$\mathrm{cut}(x) = \sum_{(i,j,w)\in E} w\,\mathbb{1}[x_i \neq x_j] = \tfrac12 \sum_e w_e\,(1 - x_i x_j).$$

Max-Cut is invariant under flipping every spin ($x\to-x$ just swaps the two sides), so the
certificate is only defined up to that $\mathbb{Z}_2$ gauge; we fix it by choosing $x_0=+1$.

**The instances.** We do not redistribute the Gset graphs — they are the standard public
benchmark set. Point `GSET_DIR` at your own copy, or run the optional download cell to fetch
them from the canonical source.

In [ ]:
import json, urllib.request
from pathlib import Path
import numpy as np
import pandas as pd

HERE = Path.cwd()                                   # Jupyter runs from this notebook's dir
SOLUTIONS   = HERE / ".." / "solutions" / "gset.json"
LEADERBOARD = HERE / ".." / ".." / "data" / "gset.csv"

# Where the Gset instance files live. We do NOT ship them (standard public benchmark set).
# Point this at your local copy, or run the optional download cell below.
GSET_DIR = HERE / "gset_instances"
GSET_URL = "https://web.stanford.edu/~yyye/yyye/Gset/{instance}"   # canonical public source

doc  = json.loads(Path(SOLUTIONS).read_text())
sols = doc["solutions"]
print(doc["solver"], "\u2014", len(sols), "certificates")
print(doc["convention"])

### (optional) fetch the Gset instances
Run this only if you don't already have the Gset files locally. It downloads any missing
instance into `GSET_DIR` from the canonical public source. Skip it and set `GSET_DIR` to your
own copy if you prefer.

In [ ]:
GSET_DIR.mkdir(parents=True, exist_ok=True)
for inst in sols:
    f = GSET_DIR / inst
    if not f.exists():
        urllib.request.urlretrieve(GSET_URL.format(instance=inst), f)
        print("downloaded", inst)
print("instances present:", sum((GSET_DIR / i).exists() for i in sols), "/", len(sols))

In [ ]:
def parse_gset(path):
    """Gset edge list -> 0-indexed (i, j, w). First line 'N edges'; rows 'u v w' are 1-indexed."""
    with open(path) as fh:
        fh.readline()
        e = np.array([[int(t) for t in ln.split()] for ln in fh if ln.strip()])
    return e[:, 0] - 1, e[:, 1] - 1, e[:, 2]

def cut_of(bits, i, j, w):
    """cut(x) = 1/2 * sum_e w_e (1 - x_i x_j), with x_i=+1 for bit '1', -1 for '0'."""
    x = np.where(np.frombuffer(bits.encode(), np.uint8) == ord("1"), 1, -1)
    return int(0.5 * (w.sum() - (x[i] * x[j]) @ w))

rows = []
for inst, s in sols.items():
    f = GSET_DIR / inst
    if not f.exists():
        rows.append({"instance": inst, "N": s["N"], "claimed": s["cut"],
                     "recomputed": None, "match": None})
        continue
    i, j, w = parse_gset(f)
    assert len(s["bits"]) == s["N"], f"{inst}: bits length != N"
    rc = cut_of(s["bits"], i, j, w)
    rows.append({"instance": inst, "N": s["N"], "claimed": s["cut"],
                 "recomputed": rc, "match": rc == s["cut"]})

In [ ]:
df = pd.DataFrame(rows)

# cross-check each claimed cut against the published leaderboard (data/gset.csv, 'cut' column)
lb = pd.read_csv(LEADERBOARD).set_index("instance")["cut"]
df["leaderboard"] = df["instance"].map(lb)
df["== leaderboard"] = df["claimed"] == df["leaderboard"]

checked = df[df["recomputed"].notna()]
print(f"verified {len(checked)}/{len(df)} instances"
      + ("" if len(checked) == len(df) else "  (others: instance file not found in GSET_DIR)"))
print("all recomputed cuts == certificate:", bool(checked["match"].all()))
print("all certificates  == leaderboard: ", bool(df["== leaderboard"].all()))
assert checked["match"].all(), "a recomputed cut disagrees with its certificate!"
assert df["== leaderboard"].all(), "a certificate disagrees with the published leaderboard!"
df